# Module 6 Project: Edge AI, Compression, MLOps, and Governance


> **Colab setup:** This notebook uses only the default open-source Python stack available in Google Colab: NumPy, Matplotlib, and scikit-learn. No `pip install`, no API keys, and no external authorization are required.
>
> **How to work:** Complete only the `TODO` lines. Most cells are partially written so you practice the key ideas without building everything from scratch.


## Learning goals
- Quantize a simple classifier
- Compare model size and accuracy
- Benchmark batch latency
- Compute a simple drift score
- Create a reproducible configuration fingerprint


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time, json, hashlib
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

np.random.seed(7)
digits = load_digits()
X = digits.images.reshape(len(digits.images), -1) / 16.0
y = digits.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=7, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
clf = LogisticRegression(max_iter=800)
clf.fit(X_train_s, y_train)
base_pred = clf.predict(X_test_s)
print("baseline accuracy:", accuracy_score(y_test, base_pred))


In [ ]:
# TODO 1: complete symmetric int8 quantization of linear weights.
W = clf.coef_.copy()
b = clf.intercept_.copy()
scale = np.max(np.abs(W)) / ____
W_q = np.round(W / scale).astype(np.int8)
W_deq = ____

logits = X_test_s @ W_deq.T + b
q_pred = logits.argmax(axis=1)
print("quantized accuracy:", accuracy_score(y_test, q_pred))
print("float bytes:", W.nbytes)
print("int8 bytes:", W_q.nbytes)


In [ ]:
# TODO 2: benchmark latency for different batch sizes.
batch_sizes = [1, 8, 32, 128]
latencies = []
for bs in batch_sizes:
    batch = X_test_s[:bs]
    start = time.perf_counter()
    for _ in range(200):
        _ = batch @ W_deq.T + b
    elapsed = time.perf_counter() - start
    latencies.append(____)

plt.plot(batch_sizes, latencies, marker="o")
plt.title("Average inference latency per batch")
plt.xlabel("batch size")
plt.ylabel("milliseconds")
plt.grid(True)
plt.show()


In [ ]:
# TODO 3: compute a simple population stability index.
def psi(expected, actual, bins=10):
    cuts = np.percentile(expected, np.linspace(0, 100, bins + 1))
    cuts = np.unique(cuts)
    e_counts, _ = np.histogram(expected, bins=cuts)
    a_counts, _ = np.histogram(actual, bins=cuts)
    e = e_counts / max(e_counts.sum(), 1)
    a = a_counts / max(a_counts.sum(), 1)
    e = np.clip(e, 1e-6, 1)
    a = np.clip(a, 1e-6, 1)
    return ____

train_brightness = X_train.mean(axis=1)
shifted = np.clip(X_test + 0.25, 0, 1)
shift_brightness = shifted.mean(axis=1)
drift_score = psi(train_brightness, shift_brightness)
print("PSI drift score:", drift_score)

plt.hist(train_brightness, alpha=0.6, label="train")
plt.hist(shift_brightness, alpha=0.6, label="shifted")
plt.legend()
plt.title("Brightness distribution drift")
plt.show()


In [ ]:
# TODO 4: create a governance fingerprint.
config = {
    "dataset": "sklearn_digits",
    "model": "logistic_regression",
    "preprocessing": "StandardScaler",
    "quantization": "symmetric_int8",
    "random_seed": 7,
}
payload = json.dumps(config, sort_keys=True)
fingerprint = hashlib.sha256(____).hexdigest()
print(fingerprint)


In [ ]:
# TODO 5: define a rollout decision gate.
accuracy_drop = accuracy_score(y_test, base_pred) - accuracy_score(y_test, q_pred)
max_allowed_drop = 0.02
max_allowed_psi = 0.25
deploy_ok = (____ <= max_allowed_drop) and (____ <= max_allowed_psi)
print("accuracy_drop:", accuracy_drop)
print("drift_score:", drift_score)
print("deploy_ok:", deploy_ok)
